# Skimming

The ntuples are very large, especially for data, and most of their content is background.
For reducing their size, it is advisable to perform an initial selection based on trigger and kinematic variables.
We will start with the trigger selection and search for useful kinematic variables later.

To guide us in the selection we will use the signal Monte Carlo sample, which has been selected as the same as the data.
Data sidebands will be used to represent the background that we want to suppress.

## Online Selection

In the online selection of the events, at HLT2 level, it is required that our signal candidates (`Hlt2RD_TauToMuMuMu`) satisfy the following conditions

- $\tau\to µµµ$
    - track candidates: PIDµ > 0, GhostProbability < 0.5, $\chi^2_{\text{trk}}$<4
    - at least 2 muons satisfying IS_MUON
    - 0 < $m_{µ_1µ_2}$ < 1800 MeV/$c^2$
    - $\chi^2_{\text{IP}}$ < 3000

The control channel (`Hlt2RD_DsToPhiPi_PhiToMuMu`) is instead selected by 

- $D_s\to\phi(1020)(\to µµ)\pi$:
    - µ: IS_MUON, PIDµ > 0, GhostProbability < 0.5, $\chi^2_{\text{trk}}$<4
    - π: GhostProbability < 0.5, $\chi^2_{\text{trk}}$<4
    - 1718 MeV/$c^2$ < $m_{µµπ}$ < 2218 MeV/$c^2$
    - 950 MeV/$c^2$ < $m_{µ_1µ_2}$ < 1090 MeV/$c^2$
    - $\chi^2_{\text{IP}}$ < 3000

### 2-µ vs. 3-µ Sample

As can be seen from the $\tau\to µµµ$ selection, `IS_MUON`, i.e. specific binary selection based on information from the muon stations

- IS_MUON
    - p < 3 GeV/c: Never classified as a muon.
    - 3 < p < 6 GeV/c: Requires hits in M2 & M3.
    - 6 < p < 10 GeV/c: Requires hits in M2 & M3 & (M4 || M5).
    - p > 10 GeV/c: Requires hits in M2 & M3 & M4 & M5

is required for at least two of the three muons. This allow us to identify two categories of selected events: those for which all 3 muons are identified as `IS_MUON`, and those with only 2 out of 3 muons identified.


In [2]:
import os
import uproot
import yaml
from importlib.resources import files

In [4]:
data_samples = yaml.safe_load(open('../data/testing_ap.yaml', 'r'))
data = uproot.open(data_samples['data'][0])

In [ ]:
keys = data['Tau2MuMuMu/DecayTree'].keys()
muon_vars = [k for k in keys if 'ISMUON' in k]
print(muon_vars)

Variabili IS_MUON trovate: ['mu1_ISMUON', 'mu2_ISMUON', 'mu3_ISMUON']


In [15]:
variables = ['Tau_M', 'Tau_PT', 'mu1_ISMUON', 'mu2_ISMUON', 'mu3_ISMUON']
df = data['Tau2MuMuMu/DecayTree'].arrays(variables, library="pd")
print(df)

              Tau_M       Tau_PT  mu1_ISMUON  mu2_ISMUON  mu3_ISMUON
0       1751.695543  2969.326172        True        True       False
1       1855.624716   102.541176        True       False        True
2       1773.077767  1252.556030       False        True        True
3       1748.234952  4443.276855        True       False        True
4       1627.707621   939.184509        True       False        True
...             ...          ...         ...         ...         ...
992665  1881.958404   694.559204        True        True       False
992666  1775.542104  4037.447510        True       False        True
992667  1606.401250  1910.755249       False        True        True
992668  1972.331368  1402.435181        True        True        True
992669  1596.410124   980.686035       False        True        True

[992670 rows x 5 columns]


In [29]:
separator = df[['mu1_ISMUON', 'mu2_ISMUON', 'mu3_ISMUON']].astype(int).sum(axis=1)
df_3mu = df[separator==3]
df_2mu = df[separator==2]
print(df_3mu)
print(df_2mu)

              Tau_M       Tau_PT  mu1_ISMUON  mu2_ISMUON  mu3_ISMUON
16      1716.931783  1882.387085        True        True        True
42      1976.821869  2430.517334        True        True        True
56      1694.052761  1296.818481        True        True        True
65      1748.263000  2400.692139        True        True        True
66      1831.606852  2211.192383        True        True        True
...             ...          ...         ...         ...         ...
992579  1841.714943   553.335876        True        True        True
992618  1739.659501  2738.827881        True        True        True
992636  1764.941298  1010.810120        True        True        True
992637  1710.825310  1936.445557        True        True        True
992668  1972.331368  1402.435181        True        True        True

[66385 rows x 5 columns]
              Tau_M       Tau_PT  mu1_ISMUON  mu2_ISMUON  mu3_ISMUON
0       1751.695543  2969.326172        True        True       False
1       

## Trigger Selection

LHC collides protons at 13.6TeV center-of-mass energy with a rate of 40 MHz.
LHCb selects events online through a 2-step approach:

1. Hlt1: a GPU farm performs initial reconstruction of the data and reduces the data rate from 40MHz to up to 1.5 MHz
2. Hlt2: a CPU farm performs the full reconstruction of the data, runs most of the selections of particle candidates and reduces the data rate to about 30KHz, corresponding to 10Gb/s stored to disk.

![RTA dataflow](https://lhcb-glance.cern.ch/alcm/public/figure/details/32#:~:text=RTA_dataflow_widescreen.pdf)

Since our candidates selected at HLT2 may have passed various selections at HLT1, we want to identify those that are most efficient to reduce background levels and define a specific trigger sequence to simplify the efficiency determination.

To determine whether the candidate has been selected by a specific Hlt1 line or not, the following variables are stored in the tuples:

- `Hlt1_<name>Decision_TOS`: Trigger On Signal, the tracks used to build our particle candidate fired the trigger line
- `Hlt1_<name>Decision_TIS`: Trigger Independently of Signal, the tracks used to build our particle candidate were not involved in the firing of the trigger line